In [1]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 1s (19.3 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 125080 files and direc

# 데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumBarunGothic')

!pip install tspoon
import tspoon as tsp

In [3]:
!pip install xmltodict
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom
import xmltodict

import json
from urllib.request import urlopen

def getECOS(topic,period,beg,end,item1='?',item2='?',item3='?',item4='?', df=None):
    key = ''
    url = 'https://ecos.bok.or.kr/api/StatisticSearch/'+'OIT9OYFOZDZWLHSBQJWU'+'/xml/kr/'+str(1)+'/'+str(50000)+'/'+topic+'/'+period+'/'+beg+'/'+end \
            +'/'+item1+'/'+item2+'/'+item3+'/'+item4
    # print(url)
    ## call OpenAPI URL
    response = requests.get(url)

    ## get API return value upon the http request
    if response.status_code == 200:
        try:
            contents = response.text
            ecosRoot = ET.fromstring(contents)

            # error check
            if ecosRoot[0].text[:4] in ("INFO","ERRO"):
                print(ecosRoot[0].text + " : " + ecosRoot[1].text)

            else:
                #print(contents)
                #dom = xml.dom.minidom.parse(xml_fname)
                dom = xml.dom.minidom.parseString(contents)
                pretty_xml_as_string = dom.toprettyxml(indent=" ")
                # print(pretty_xml_as_string)
                dic = xmltodict.parse(pretty_xml_as_string)

                n = int(dic['StatisticSearch']['list_total_count']['#text'])
                df_ecos = pd.DataFrame(index = [dic['StatisticSearch']['row'][i]['TIME'] for i in range(n)])
                df_ecos[dic['StatisticSearch']['row'][0]['ITEM_NAME1']] = [float(dic['StatisticSearch']['row'][i]['DATA_VALUE']) for i in range(n)]

                if type(df)==type(pd.DataFrame()):
                    df_ecos = df.merge(df_ecos, left_index=True, right_index=True, how='left')

                return df_ecos

        except Exception as e:
            print(str(e))


def getKOSIS(table,period,beg,end,item, orgId='101', obj1='',obj2='',obj3='',obj4='',obj5='',obj6='',obj7='',obj8='', title=None, title_no='0', debug=False):
     # Korean Statistics
    key = 'MzhiMmQwZTYyMWZiNGM1ZGFkZDJmZTI2OTE1OGU2NzQ='
    url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey='+key \
            +'&orgId='+orgId\
            +'&tblId='+table \
            +'&prdSe='+period \
            +'&startPrdDe='+beg \
            +'&endPrdDe='+end \
            +'&itmId='+item \
            +'&objL1='+obj1 \
            +'&objL2='+obj2 \
            +'&objL3='+obj3 \
            +'&objL4='+obj4 \
            +'&objL5='+obj5 \
            +'&objL6='+obj6 \
            +'&objL7='+obj7 \
            +'&objL8='+obj8 \
            +'&format=json&jsonVD=Y&outputFields=ITM_NM+PRD_DE+DT'
    res = requests.get(url)
    py_json = res.json()

    data = []
    for v in py_json:
        value = []
        value.append(v['PRD_DE'])

        # ✅ DT 값이 '-' 또는 ''일 때 NaN으로 처리
        try:
            value.append(float(v['DT']))
        except (ValueError, KeyError):
            value.append(np.nan)

        data.append(value)

    df = pd.DataFrame(data, columns=['PRD_DE', 'DT'])

    if debug:
        print(df.head())

    return df


    #get json data from url
    with urlopen(url) as url_:
        json_file = url_.read()

    py_json = json.loads(json_file.decode('utf-8'))


    #data transformation
    data = []

    for i, v in enumerate(py_json):
        value = []
        value.append(v['PRD_DE'])
        value.append(float(v['DT']))

        data.append(value)

    if title is not None:
        title = str(title)
    elif title_no=='0':
        title = v['ITM_NM_ENG']
    else:
        title = v['C'+title_no+'_NM_ENG']

    df_kosis = pd.DataFrame({v['PRD_SE']:[x[0] for x in data], title:[x[1] for x in data]}).set_index(v['PRD_SE'])

    if debug:
        return v

    return df_kosis

In [4]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

In [5]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [6]:
# change directory to where you have your data file!!
%cd gdrive/MyDrive/GDP/데이터

/content/gdrive/MyDrive/GDP/데이터


## GDP

"('200Y104', 'Q', '1980Q1', '2025Q1','110401')
+ ('200Y104', 'Q', '1980Q1', '2025Q1','110402')"
('200Y104', 'Q', '1980Q1', '2025Q1','110403')

In [8]:
gdp1 = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','110401').reset_index()
gdp2 = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','110402').reset_index()
gdp3 = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','110403').reset_index()


In [11]:
from functools import reduce
# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [gdp1,gdp2,gdp3]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
gdp = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
gdp



,index,전기업,"가스, 증기 및 공기조절 공급업","수도, 하수 및 폐기물 처리, 원료 재생업"
0,2014Q1,4576.9,1476.4,3167.7
1,2014Q2,4656.3,1483.6,3179.3
2,2014Q3,4713.4,1486.3,3212.1
3,2014Q4,5050.0,1456.1,3204.8
4,2015Q1,5138.3,1390.5,3321.0
5,2015Q2,5173.4,1436.3,3369.4
6,2015Q3,5339.8,1397.4,3439.0
7,2015Q4,5524.6,1273.4,3284.0
8,2016Q1,5252.0,1392.8,3432.7
9,2016Q2,5068.1,1345.3,3558.8


In [12]:
gdp['gdp'] = gdp['전기업']+gdp['가스, 증기 및 공기조절 공급업']+gdp['수도, 하수 및 폐기물 처리, 원료 재생업']
gdp = gdp.drop(columns=["전기업","가스, 증기 및 공기조절 공급업","수도, 하수 및 폐기물 처리, 원료 재생업"])

In [13]:
gdp

,index,gdp
0,2014Q1,9221.0
1,2014Q2,9319.2
2,2014Q3,9411.8
3,2014Q4,9710.9
4,2015Q1,9849.8
5,2015Q2,9979.1
6,2015Q3,10176.2
7,2015Q4,10082.0
8,2016Q1,10077.5
9,2016Q2,9972.2


##PPI(단순평균)

('404Y014','Q','1965Q1','2025Q2','401AA')
('404Y014','Q','1965Q1', '2025Q2','402AA')

In [14]:
ppi1 = getECOS('404Y014','Q','2014Q1','2025Q2','401AA').reset_index()
ppi2 = getECOS('404Y014','Q','2014Q1','2025Q2','402AA').reset_index()

In [16]:
# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [ppi1,ppi2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
ppi = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
ppi

ppi['ppi']=(ppi['전력,가스및증기']+ppi['수도,폐기물처리및재활용서비스'])/2
ppi = ppi.drop(columns=["전력,가스및증기","수도,폐기물처리및재활용서비스"])
ppi

,index,ppi
0,2014Q1,97.340
1,2014Q2,97.745
2,2014Q3,97.725
3,2014Q4,97.825
4,2015Q1,95.625
5,2015Q2,92.665
6,2015Q3,91.925
7,2015Q4,92.945
8,2016Q1,92.080
9,2016Q2,90.820


##BSI(단순평균)

D3500, S5000

In [17]:
업황실적BSI1 = getECOS('512Y007','M','201401','202506','AA','D3500').reset_index()
업황실적BSI2 = getECOS('512Y007','M','201401','202506','AA','S5000').reset_index()

In [18]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI1,업황실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
업황실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
업황실적BSI

,index,업황실적BSI 1)_x,업황실적BSI 1)_y
0,201401,105.0,60.0
1,201402,91.0,53.0
2,201403,95.0,61.0
3,201404,87.0,64.0
4,201405,82.0,69.0
...,...,...,...
133,202502,91.0,51.0
134,202503,95.0,52.0
135,202504,83.0,52.0
136,202505,85.0,58.0


In [19]:
업황실적BSI['업황실적BSI']=(업황실적BSI['업황실적BSI 1)_x']+업황실적BSI['업황실적BSI 1)_y'])/2
업황실적BSI = 업황실적BSI.drop(columns=["업황실적BSI 1)_x","업황실적BSI 1)_y"])
업황실적BSI

,index,업황실적BSI
0,201401,82.5
1,201402,72.0
2,201403,78.0
3,201404,75.5
4,201405,75.5
...,...,...
133,202502,71.0
134,202503,73.5
135,202504,67.5
136,202505,71.5


In [20]:
매출실적BSI1 = getECOS('512Y007','M','201401','202506','AB','D3500').reset_index()
매출실적BSI2 = getECOS('512Y007','M','201401','202506','AB','S5000').reset_index()

from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [매출실적BSI1,매출실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
매출실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
매출실적BSI

매출실적BSI['매출실적BSI']=(매출실적BSI['매출실적BSI 2)_x']+매출실적BSI['매출실적BSI 2)_y'])/2
매출실적BSI = 매출실적BSI.drop(columns=["매출실적BSI 2)_x","매출실적BSI 2)_y"])
매출실적BSI

,index,매출실적BSI
0,201401,75.5
1,201402,79.5
2,201403,80.5
3,201404,79.5
4,201405,75.5
...,...,...
133,202502,72.0
134,202503,78.0
135,202504,73.0
136,202505,76.0


In [21]:
채산성실적BSI1 = getECOS('512Y007','M','201401','202506','AE','D3500').reset_index()
채산성실적BSI2 = getECOS('512Y007','M','201401','202506','AE','S5000').reset_index()
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [채산성실적BSI1,채산성실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
채산성실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
채산성실적BSI

채산성실적BSI['채산성실적BSI']=(채산성실적BSI['채산성실적BSI 6)_x']+채산성실적BSI['채산성실적BSI 6)_y'])/2
채산성실적BSI = 채산성실적BSI.drop(columns=["채산성실적BSI 6)_x","채산성실적BSI 6)_y"])
채산성실적BSI

,index,채산성실적BSI
0,201401,87.5
1,201402,76.0
2,201403,78.5
3,201404,86.0
4,201405,83.0
...,...,...
133,202502,75.0
134,202503,78.5
135,202504,78.0
136,202505,78.5


In [22]:
자금사정실적BSI1 = getECOS('512Y007','M','201401','202506','AO','D3500').reset_index()
자금사정실적BSI2 = getECOS('512Y007','M','201401','202506','AO','S5000').reset_index()


# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [자금사정실적BSI1,자금사정실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
자금사정실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
자금사정실적BSI


자금사정실적BSI['자금사정실적BSI']=(자금사정실적BSI['자금사정실적BSI 6)_x']+자금사정실적BSI['자금사정실적BSI 6)_y'])/2
자금사정실적BSI = 자금사정실적BSI.drop(columns=["자금사정실적BSI 6)_x","자금사정실적BSI 6)_y"])
자금사정실적BSI

,index,자금사정실적BSI
0,201401,86.5
1,201402,81.0
2,201403,77.5
3,201404,83.5
4,201405,86.0
...,...,...
133,202502,78.0
134,202503,78.5
135,202504,76.0
136,202505,77.0


In [24]:
인력사정실적BSI1 = getECOS('512Y007','M','201401','202506','AJ','D3500').reset_index()
인력사정실적BSI2 = getECOS('512Y007','M','201401','202506','AJ','S5000').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [인력사정실적BSI1,인력사정실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
인력사정실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
인력사정실적BSI

인력사정실적BSI['인력사정실적BSI']=(인력사정실적BSI['인력사정실적BSI 3)_x']+인력사정실적BSI['인력사정실적BSI 3)_y'])/2
인력사정실적BSI = 인력사정실적BSI.drop(columns=["인력사정실적BSI 3)_x","인력사정실적BSI 3)_y"])
인력사정실적BSI

,index,인력사정실적BSI
0,201401,86.0
1,201402,85.0
2,201403,87.0
3,201404,88.5
4,201405,87.0
...,...,...
133,202502,74.0
134,202503,80.0
135,202504,82.5
136,202505,78.5


In [25]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI, 매출실적BSI, 채산성실적BSI, 자금사정실적BSI, 인력사정실적BSI]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
df_merged = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
df_merged

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI
0,201401,82.5,75.5,87.5,86.5,86.0
1,201402,72.0,79.5,76.0,81.0,85.0
2,201403,78.0,80.5,78.5,77.5,87.0
3,201404,75.5,79.5,86.0,83.5,88.5
4,201405,75.5,75.5,83.0,86.0,87.0
...,...,...,...,...,...,...
133,202502,71.0,72.0,75.0,78.0,74.0
134,202503,73.5,78.0,78.5,78.5,80.0
135,202504,67.5,73.0,78.0,76.0,82.5
136,202505,71.5,76.0,78.5,77.0,78.5


In [26]:
df_merged.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
채산성실적BSI,0
자금사정실적BSI,0
인력사정실적BSI,0


In [27]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged['index'] = df_merged['index'].astype(str)

# 연도와 월 분리
df_merged['연도'] = df_merged['index'].str[:4].astype(int)
df_merged['월'] = df_merged['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged['분기'] = df_merged['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged['index'] = df_merged['연도'].astype(str) + df_merged['분기']

# 분기별 평균 집계
df_quarter = df_merged.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter.head())

    index    업황실적BSI    매출실적BSI   채산성실적BSI  자금사정실적BSI  인력사정실적BSI      연도     월
0  2014Q1  77.500000  78.500000  80.666667  81.666667  86.000000  2014.0   2.0
1  2014Q2  73.500000  74.833333  82.833333  84.333333  88.666667  2014.0   5.0
2  2014Q3  69.833333  72.666667  77.500000  78.500000  89.666667  2014.0   8.0
3  2014Q4  70.333333  75.333333  81.500000  83.000000  91.166667  2014.0  11.0
4  2015Q1  65.000000  72.666667  77.166667  78.333333  90.666667  2015.0   2.0


In [28]:
df_quarter = df_quarter.drop(columns=['연도', '월'], errors='ignore')

In [29]:
dfs = [ppi, gdp]
df_final = df_quarter.copy()
key_col = 'index'
for df in dfs:
    df_final = df_final.merge(df, on=key_col, how='left')

df_final

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp
0,2014Q1,77.500000,78.500000,80.666667,81.666667,86.000000,97.340,9221.0
1,2014Q2,73.500000,74.833333,82.833333,84.333333,88.666667,97.745,9319.2
2,2014Q3,69.833333,72.666667,77.500000,78.500000,89.666667,97.725,9411.8
3,2014Q4,70.333333,75.333333,81.500000,83.000000,91.166667,97.825,9710.9
4,2015Q1,65.000000,72.666667,77.166667,78.333333,90.666667,95.625,9849.8
5,2015Q2,73.000000,71.333333,82.833333,85.166667,85.833333,92.665,9979.1
6,2015Q3,62.166667,58.500000,75.666667,80.000000,85.666667,91.925,10176.2
7,2015Q4,61.833333,57.000000,75.833333,80.833333,87.500000,92.945,10082.0
8,2016Q1,62.166667,61.166667,76.666667,79.333333,87.333333,92.080,10077.5
9,2016Q2,66.833333,65.500000,79.000000,83.333333,84.500000,90.820,9972.2


In [30]:
df_final.to_csv("전기가스수도1차.csv")

## 생산지수

('DT_1F02001','Q','197501','202502','T20','101','00','D35')
('DT_1KC2020','Q','200001','202502','T3','101','E')

In [39]:
생산지수 = getKOSIS('DT_1KC2020','Q','201401','202502','T3','101','E')
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수

,생산지수,index
0,87.7,2014Q1
1,85.3,2014Q2
2,86.9,2014Q3
3,87.8,2014Q4
4,88.6,2015Q1
5,90.9,2015Q2
6,94.3,2015Q3
7,90.1,2015Q4
8,92.3,2016Q1
9,95.1,2016Q2


https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T20+&objL1=D+&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Q&startPrdDe=201401&endPrdDe=202502&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=101&tblId=DT_1F02016

## 내수출하지수

In [40]:
내수출하지수 = getKOSIS('DT_1F02016','Q','201401','202502','T20','101','D')

In [41]:
def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

내수출하지수['index']=내수출하지수['PRD_DE'].apply(to_quarter)
내수출하지수 = 내수출하지수.drop(columns=["PRD_DE"])
내수출하지수 = 내수출하지수.rename(columns={'DT': '내수출하지수'})
내수출하지수

,내수출하지수,index
0,91.701,2014Q1
1,94.490,2014Q2
2,93.042,2014Q3
3,95.707,2014Q4
4,94.011,2015Q1
5,92.986,2015Q2
6,92.834,2015Q3
7,90.764,2015Q4
8,95.550,2016Q1
9,92.520,2016Q2


## 기업데이터

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [43]:
df = pd.read_csv('임지오기업.csv')

전기업 + 가스, 증기 및 공기조절 공급업 수도, 하수 및 폐기물 처리, 원료 재생업

In [44]:
운송장비기업1 = df[df['매핑한 산업'] == "전기업 + 가스, 증기 및 공기조절 공급업"]
운송장비기업2 = df[df['매핑한 산업'] == "하수 및 폐기물 처리, 원료 재생업"]

In [45]:
columns = ['EBITDA', '인건비']
grouped1 = 운송장비기업1.groupby('조사일')[columns].sum()
grouped1

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,1.001799e+08
2000-04-01,0.000000e+00,1.270041e+08
2000-07-01,0.000000e+00,9.326003e+07
2000-10-01,9.611912e+12,1.335659e+09
2001-01-01,0.000000e+00,9.946722e+07
...,...,...
2024-04-01,1.793114e+12,2.443646e+08
2024-07-01,1.090854e+12,2.565800e+08
2024-10-01,5.366528e+12,2.415679e+08


In [46]:
grouped1 = grouped1.reset_index()

In [47]:
grouped1['조사일']=pd.to_datetime(grouped1['조사일'])

# 분기(Quarter) 단위로 변환
grouped1["index"] = grouped1["조사일"].dt.to_period("Q")
grouped1 = grouped1.drop(columns=["조사일"])
print(grouped1)

grouped1

           EBITDA           인건비   index
0    0.000000e+00  1.001799e+08  2000Q1
1    0.000000e+00  1.270041e+08  2000Q2
2    0.000000e+00  9.326003e+07  2000Q3
3    9.611912e+12  1.335659e+09  2000Q4
4    0.000000e+00  9.946722e+07  2001Q1
..            ...           ...     ...
97   1.793114e+12  2.443646e+08  2024Q2
98   1.090854e+12  2.565800e+08  2024Q3
99   5.366528e+12  2.415679e+08  2024Q4
100  4.604242e+12  2.041978e+08  2025Q1
101  2.828906e+12 -1.913964e+08  2025Q2

[102 rows x 3 columns]


,EBITDA,인건비,index
0,0.000000e+00,1.001799e+08,2000Q1
1,0.000000e+00,1.270041e+08,2000Q2
2,0.000000e+00,9.326003e+07,2000Q3
3,9.611912e+12,1.335659e+09,2000Q4
4,0.000000e+00,9.946722e+07,2001Q1
...,...,...,...
97,1.793114e+12,2.443646e+08,2024Q2
98,1.090854e+12,2.565800e+08,2024Q3
99,5.366528e+12,2.415679e+08,2024Q4
100,4.604242e+12,2.041978e+08,2025Q1


In [51]:
mask = (grouped1["index"] >= pd.Period("2014Q1")) & (grouped1["index"] <= pd.Period("2025Q2"))
기업= grouped1.loc[mask]
기업
기업['합산'] = 기업['EBITDA']+기업['인건비']
기업



/tmp/ipython-input-3670662141.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  기업['합산'] = 기업['EBITDA']+기업['인건비']


,EBITDA,인건비,index,합산
56,1.574491e+12,190183934.0,2014Q1,1.574682e+12
57,8.579764e+11,186647701.0,2014Q2,8.581630e+11
58,2.455023e+12,194933769.0,2014Q3,2.455218e+12
59,1.842156e+12,159067852.0,2014Q4,1.842315e+12
60,1.940467e+12,201482251.0,2015Q1,1.940669e+12
61,3.114885e+12,200631918.0,2015Q2,3.115086e+12
62,3.111111e+12,194980600.0,2015Q3,3.111306e+12
63,1.698320e+12,245019534.0,2015Q4,1.698565e+12
64,2.543067e+12,211014421.0,2016Q1,2.543278e+12
65,2.935296e+12,196393733.0,2016Q2,2.935493e+12


## 전체 데이터 합산

In [52]:
from functools import reduce
import pandas as pd

# 모든 데이터프레임의 index 컬럼을 문자열로 변환
for df in [df_final,생산지수, 내수출하지수, 기업]:
    df['index'] = df['index'].astype(str)

# 리스트에 넣고 합치기
dfs = [df_final,생산지수, 내수출하지수, 기업]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)


/tmp/ipython-input-4156791558.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['index'] = df['index'].astype(str)


In [53]:
df_merged
df_merged = df_merged[['index'] + [col for col in df_merged.columns if col != 'index']]

In [54]:
df_merged

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,내수출하지수,EBITDA,인건비,합산
0,2014Q1,77.500000,78.500000,80.666667,81.666667,86.000000,97.340,9221.0,87.7,91.701,1.574491e+12,190183934.0,1.574682e+12
1,2014Q2,73.500000,74.833333,82.833333,84.333333,88.666667,97.745,9319.2,85.3,94.490,8.579764e+11,186647701.0,8.581630e+11
2,2014Q3,69.833333,72.666667,77.500000,78.500000,89.666667,97.725,9411.8,86.9,93.042,2.455023e+12,194933769.0,2.455218e+12
3,2014Q4,70.333333,75.333333,81.500000,83.000000,91.166667,97.825,9710.9,87.8,95.707,1.842156e+12,159067852.0,1.842315e+12
4,2015Q1,65.000000,72.666667,77.166667,78.333333,90.666667,95.625,9849.8,88.6,94.011,1.940467e+12,201482251.0,1.940669e+12
5,2015Q2,73.000000,71.333333,82.833333,85.166667,85.833333,92.665,9979.1,90.9,92.986,3.114885e+12,200631918.0,3.115086e+12
6,2015Q3,62.166667,58.500000,75.666667,80.000000,85.666667,91.925,10176.2,94.3,92.834,3.111111e+12,194980600.0,3.111306e+12
7,2015Q4,61.833333,57.000000,75.833333,80.833333,87.500000,92.945,10082.0,90.1,90.764,1.698320e+12,245019534.0,1.698565e+12
8,2016Q1,62.166667,61.166667,76.666667,79.333333,87.333333,92.080,10077.5,92.3,95.550,2.543067e+12,211014421.0,2.543278e+12
9,2016Q2,66.833333,65.500000,79.000000,83.333333,84.500000,90.820,9972.2,95.1,92.520,2.935296e+12,196393733.0,2.935493e+12


In [55]:
df_merged.to_csv('전기가스수도.csv')

# 데이터 QoQ 반환

In [56]:
import pandas as pd

# df는 위의 데이터프레임
df_qoq = df_merged.copy()

# 분기 순서대로 정렬 (혹시 index가 문자열일 경우를 대비)
df_qoq = df_qoq.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_qoq.select_dtypes(include=['number']).columns

# 전분기 대비 증감률 (%) 계산
df_qoq[num_cols] = df_qoq[num_cols].pct_change() * 100

# 결과 확인
print(df_qoq.head())

    index   업황실적BSI   매출실적BSI  채산성실적BSI  자금사정실적BSI  인력사정실적BSI       ppi  \
0  2014Q1       NaN       NaN       NaN        NaN        NaN       NaN   
1  2014Q2 -5.161290 -4.670913  2.685950   3.265306   3.100775  0.416067   
2  2014Q3 -4.988662 -2.895323 -6.438632  -6.916996   1.127820 -0.020461   
3  2014Q4  0.715990  3.669725  5.161290   5.732484   1.672862  0.102328   
4  2015Q1 -7.582938 -3.539823 -5.316973  -5.622490  -0.548446 -2.248914   

        gdp      생산지수    내수출하지수      EBITDA        인건비          합산  
0       NaN       NaN       NaN         NaN        NaN         NaN  
1  1.064960 -2.736602  3.041406  -45.507711  -1.859375  -45.502439  
2  0.993648  1.875733 -1.532437  186.141135   4.439416  186.101616  
3  3.177926  1.035673  2.864298  -24.963822 -18.399027  -24.963301  
4  1.430351  0.911162 -1.772075    5.336775  26.664344    5.338617  


In [57]:
df_qoq
df_qoq = df_qoq.drop(index=[0, 1, 2,3])

In [58]:
df_qoq.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
채산성실적BSI,0
자금사정실적BSI,0
인력사정실적BSI,0
ppi,0
gdp,0
생산지수,0
내수출하지수,0


In [59]:
df_qoq.to_csv('전기가스수도qoq.csv')

## 데이터YoY변환

In [60]:
import pandas as pd

# df_merged는 원본 데이터프레임
df_yoy = df_merged.copy()

# 분기 순서대로 정렬
df_yoy = df_yoy.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_yoy.select_dtypes(include=['number']).columns

# 전년 동분기 대비 증감률 (%) 계산 (4분기 차이)
df_yoy[num_cols] = df_yoy[num_cols].pct_change(periods=4) * 100

# 결과 확인
print(df_yoy.head(10))

    index    업황실적BSI    매출실적BSI  채산성실적BSI  자금사정실적BSI  인력사정실적BSI       ppi  \
0  2014Q1        NaN        NaN       NaN        NaN        NaN       NaN   
1  2014Q2        NaN        NaN       NaN        NaN        NaN       NaN   
2  2014Q3        NaN        NaN       NaN        NaN        NaN       NaN   
3  2014Q4        NaN        NaN       NaN        NaN        NaN       NaN   
4  2015Q1 -16.129032  -7.430998 -4.338843  -4.081633   5.426357 -1.761866   
5  2015Q2  -0.680272  -4.677060  0.000000   0.988142  -3.195489 -5.197197   
6  2015Q3 -10.978520 -19.495413 -2.365591   1.910828  -4.460967 -5.935022   
7  2015Q4 -12.085308 -24.336283 -6.952965  -2.610442  -4.021938 -4.988500   
8  2016Q1  -4.358974 -15.825688 -0.647948   1.276596  -3.676471 -3.707190   
9  2016Q2  -8.447489  -8.177570 -4.627767  -2.152642  -1.553398 -1.991043   

        gdp      생산지수    내수출하지수      EBITDA        인건비          합산  
0       NaN       NaN       NaN         NaN        NaN         NaN  
1       NaN   

In [61]:
df_yoy=df_yoy.dropna()

In [62]:
df_yoy

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,내수출하지수,EBITDA,인건비,합산
4,2015Q1,-16.129032,-7.430998,-4.338843,-4.081633,5.426357,-1.761866,6.819217,1.026226,2.519056,2.324408e+01,5.940732,2.324199e+01
5,2015Q2,-0.680272,-4.677060,0.000000,0.988142,-3.195489,-5.197197,7.081080,6.565064,-1.591703,2.630502e+02,7.492306,2.629947e+02
6,2015Q3,-10.978520,-19.495413,-2.365591,1.910828,-4.460967,-5.935022,8.121720,8.515535,-0.223555,2.672428e+01,0.024024,2.672216e+01
7,2015Q4,-12.085308,-24.336283,-6.952965,-2.610442,-4.021938,-4.988500,3.821479,2.619590,-5.164721,-7.808017e+00,54.034603,-7.802678e+00
8,2016Q1,-4.358974,-15.825688,-0.647948,1.276596,-3.676471,-3.707190,2.311722,4.176072,1.637042,3.105437e+01,4.731022,3.105164e+01
9,2016Q2,-8.447489,-8.177570,-4.627767,-2.152642,-1.553398,-1.991043,-0.069145,4.620462,-0.501151,-5.765507e+00,-2.112418,-5.765272e+00
10,2016Q3,11.260054,21.937322,6.607930,5.833333,1.361868,-2.273593,-1.336452,-1.060445,2.099446,1.947663e+01,22.576859,1.947682e+01
11,2016Q4,19.137466,40.935673,17.582418,8.865979,3.428571,-1.495508,-0.453283,3.884573,2.723547,-1.367340e+01,16.508593,-1.366904e+01
12,2017Q1,31.903485,42.506812,18.695652,15.546218,-0.381679,-0.119461,2.857852,4.225352,2.234432,-4.433406e+01,-0.208610,-4.433040e+01
13,2017Q2,29.177057,42.239186,9.071730,4.400000,3.944773,1.701167,4.262851,-3.154574,8.120406,-4.921464e+01,10.905989,-4.921062e+01


In [64]:
df_yoy.to_csv("전기가스수도yoy.csv")

# 상관계수 보기(qoq)

In [65]:
qoq = pd.read_csv('전기가스수도qoq.csv')
qoq = qoq.drop(columns=["Unnamed: 0"])

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# GDP_QoQ 기준
target = 'gdp'

In [67]:
# 수치형 컬럼만 선택
num_cols = qoq.select_dtypes(include=['number']).columns

# GDP 기준 상관계수 계산
corr = qoq[num_cols].corr()[target].sort_values(ascending=False)

print("📊 GDP 대비 단순 상관계수")
print(corr)


📊 GDP 대비 단순 상관계수
gdp          1.000000
내수출하지수       0.356599
인건비          0.225684
ppi          0.050955
EBITDA       0.047757
합산           0.047483
업황실적BSI     -0.073586
생산지수        -0.102609
자금사정실적BSI   -0.113082
매출실적BSI     -0.163991
채산성실적BSI    -0.280148
인력사정실적BSI   -0.332734
Name: gdp, dtype: float64


In [69]:
# df2 사본
df2 = qoq.copy()

# target 및 feature 지정
target_col = 'gdp'
feature_cols = [col for col in df2.columns if col not in ['index', target_col]]

# 결과 저장용 데이터프레임
corr_df = pd.DataFrame(columns=['Feature', 'Lag', 'Correlation'])

# lag 1~4 생성 및 상관계수 계산
for col in feature_cols:
    for lag in range(1, 5):  # lag 1~4
        lag_col_name = f"{col}_lag{lag}"
        df2[lag_col_name] = df2[col].shift(lag)  # 실제 데이터에 lag 컬럼 추가
        corr = df2[target_col].corr(df2[lag_col_name])  # 상관계수 계산
        corr_df = pd.concat(
            [corr_df, pd.DataFrame({'Feature': [col], 'Lag': [lag], 'Correlation': [corr]})],
            ignore_index=True
        )

# 상관계수 높은 순으로 정렬
corr_df.sort_values(by='Correlation', ascending=False, inplace=True)
corr_df.reset_index(drop=True, inplace=True)

corr_df

/tmp/ipython-input-1052179002.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  corr_df = pd.concat(


,Feature,Lag,Correlation
0,자금사정실적BSI,1,0.338696
1,인건비,1,0.326937
2,채산성실적BSI,1,0.270519
3,ppi,4,0.186929
4,인력사정실적BSI,4,0.185562
5,내수출하지수,2,0.183573
6,인력사정실적BSI,1,0.169198
7,매출실적BSI,1,0.161258
8,자금사정실적BSI,4,0.153795
9,ppi,1,0.125001


In [70]:
df2

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,내수출하지수,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
0,2015Q1,-7.582938,-3.539823,-5.316973,-5.622490,-0.548446,-2.248914,1.430351,0.911162,-1.772075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015Q2,12.307692,-1.834862,7.343413,8.723404,-5.330882,-3.095425,1.312717,2.595937,-1.090298,...,NaN,NaN,26.664344,NaN,NaN,NaN,5.338617,NaN,NaN,NaN
2,2015Q3,-14.840183,-17.990654,-8.651911,-6.066536,-0.194175,-0.798576,1.975128,3.740374,-0.163465,...,NaN,NaN,-0.422039,26.664344,NaN,NaN,60.516100,5.338617,NaN,NaN
3,2015Q4,-0.536193,-2.564103,0.220264,1.041667,2.140078,1.109600,-0.925689,-4.453871,-2.229787,...,5.336775e+00,NaN,-2.816759,-0.422039,26.664344,NaN,-0.121358,60.516100,5.338617,NaN
4,2016Q1,0.539084,7.309942,1.098901,-1.855670,-0.190476,-0.930658,-0.044634,2.441731,5.273016,...,6.052243e+01,5.336775e+00,25.663545,-2.816759,-0.422039,26.664344,-45.406684,-0.121358,60.516100,5.338617
5,2016Q2,7.506702,7.084469,3.043478,5.042017,-3.244275,-1.368375,-1.044902,3.033586,-3.171115,...,-1.211848e-01,6.052243e+01,-13.878531,25.663545,-2.816759,-0.422039,49.731008,-45.406684,-0.121358,60.516100
6,2016Q3,3.491272,8.905852,2.109705,1.600000,2.761341,-1.084563,0.681896,-1.892744,2.445958,...,-4.541114e+01,-1.211848e-01,-6.928762,-13.878531,25.663545,-2.816759,15.421612,49.731008,-45.406684,-0.121358
7,2016Q4,6.506024,12.616822,10.537190,3.937008,4.222649,1.914621,-0.038844,0.321543,-1.632149,...,4.974019e+01,-4.541114e+01,21.694869,-6.928762,-13.878531,25.663545,26.632539,15.421612,49.731008,-45.406684
8,2017Q1,11.312217,8.506224,2.056075,4.166667,-3.867403,0.453279,3.280093,2.777778,4.771762,...,1.542347e+01,4.974019e+01,19.442470,21.694869,-6.928762,-13.878531,-60.552239,26.632539,15.421612,49.731008
9,2017Q2,5.284553,6.883365,-5.311355,-5.090909,0.957854,0.429488,0.306787,-4.261954,2.403644,...,2.663287e+01,1.542347e+01,-26.235647,19.442470,21.694869,-6.928762,-3.447555,-60.552239,26.632539,15.421612


In [71]:
# 숫자형 컬럼만 선택 (연도분기 제외)
numeric_df = df2.select_dtypes(include=["number"])

# 전체 상관계수
corr = numeric_df.corr()

target_corr = corr["gdp"].drop("gdp")
target_corr = target_corr.sort_values(ascending=False)
target_corr

,gdp
내수출하지수,0.356599
자금사정실적BSI_lag1,0.338696
인건비_lag1,0.326937
채산성실적BSI_lag1,0.270519
인건비,0.225684
ppi_lag4,0.186929
인력사정실적BSI_lag4,0.185562
내수출하지수_lag2,0.183573
인력사정실적BSI_lag1,0.169198
매출실적BSI_lag1,0.161258


In [72]:
target_corr.to_csv('전기가스수도순서.csv')

In [73]:
df2.dropna(inplace=True)

In [74]:
df2

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,gdp,생산지수,내수출하지수,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
4,2016Q1,0.539084,7.309942,1.098901,-1.855670,-0.190476,-0.930658,-0.044634,2.441731,5.273016,...,6.052243e+01,5.336775e+00,25.663545,-2.816759,-0.422039,26.664344,-45.406684,-0.121358,60.516100,5.338617
5,2016Q2,7.506702,7.084469,3.043478,5.042017,-3.244275,-1.368375,-1.044902,3.033586,-3.171115,...,-1.211848e-01,6.052243e+01,-13.878531,25.663545,-2.816759,-0.422039,49.731008,-45.406684,-0.121358,60.516100
6,2016Q3,3.491272,8.905852,2.109705,1.600000,2.761341,-1.084563,0.681896,-1.892744,2.445958,...,-4.541114e+01,-1.211848e-01,-6.928762,-13.878531,25.663545,-2.816759,15.421612,49.731008,-45.406684,-0.121358
7,2016Q4,6.506024,12.616822,10.537190,3.937008,4.222649,1.914621,-0.038844,0.321543,-1.632149,...,4.974019e+01,-4.541114e+01,21.694869,-6.928762,-13.878531,25.663545,26.632539,15.421612,49.731008,-45.406684
8,2017Q1,11.312217,8.506224,2.056075,4.166667,-3.867403,0.453279,3.280093,2.777778,4.771762,...,1.542347e+01,4.974019e+01,19.442470,21.694869,-6.928762,-13.878531,-60.552239,26.632539,15.421612,49.731008
9,2017Q2,5.284553,6.883365,-5.311355,-5.090909,0.957854,0.429488,0.306787,-4.261954,2.403644,...,2.663287e+01,1.542347e+01,-26.235647,19.442470,21.694869,-6.928762,-3.447555,-60.552239,26.632539,15.421612
10,2017Q3,-1.544402,0.357782,7.736944,1.149425,2.277040,0.703730,1.840863,3.148751,2.217268,...,-6.055738e+01,2.663287e+01,3.437358,-26.235647,19.442470,21.694869,5.303294,-3.447555,-60.552239,26.632539
11,2017Q4,3.921569,2.139037,-2.154399,2.083333,-0.185529,1.859915,-1.396772,-0.210526,1.838613,...,-3.443118e+00,-6.055738e+01,17.662807,3.437358,-26.235647,19.442470,95.198668,5.303294,-3.447555,-60.552239
12,2018Q1,-0.377358,4.013962,0.183486,-3.153989,2.602230,0.089714,-3.646272,1.160338,0.636698,...,5.303571e+00,-3.443118e+00,6.500059,17.662807,3.437358,-26.235647,-38.684582,95.198668,5.303294,-3.447555
13,2018Q2,-2.083333,-2.516779,0.000000,2.107280,-3.804348,0.237267,2.002962,2.189781,0.495257,...,9.521000e+01,5.303571e+00,-12.807501,6.500059,17.662807,3.437358,-45.922768,-38.684582,95.198668,5.303294


In [75]:
df2 = df2.reset_index()
df2 = df2.drop(columns=["level_0"])

In [76]:
df2.to_csv('전기가스수도lag.csv', encoding='utf-8-sig')

# ARIMA

In [77]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기가스수도lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','인력사정실적BSI','내수출하지수_lag1']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 2.7013
📈 평균 Test MAE: 2.7196


In [78]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기가스수도lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','인력사정실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (1.0, 1.0, 0.0)
📉 평균 Train MAE: 3.0967
📈 평균 Test MAE: 2.5402


In [79]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기가스수도lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','내수출하지수_lag1']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 2.8008
📈 평균 Test MAE: 2.3451


In [80]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기가스수도lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (1.0, 1.0, 0.0)
📉 평균 Train MAE: 3.1068
📈 평균 Test MAE: 2.0425


In [82]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기가스수도lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','내수출하지수_lag1']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (1.0, 1.0, 1.0)
📉 평균 Train MAE: 2.7677
📈 평균 Test MAE: 2.2884


In [83]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기가스수도lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['내수출하지수','자금사정실적BSI_lag1']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 3.0388
📈 평균 Test MAE: 2.3105


In [84]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('전기가스수도lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = 'gdp'   # GDP 예측 대상 칼럼
exog_vars = ['내수출하지수']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (2.0, 1.0, 1.0)
📉 평균 Train MAE: 2.6208
📈 평균 Test MAE: 2.3875


# 랜덤포레스트

In [85]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','인력사정실적BSI','내수출하지수_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.0618
📈 평균 Test MAE: 2.9075


In [86]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','인력사정실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.0539
📈 평균 Test MAE: 3.0462


In [87]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','내수출하지수_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.0637
📈 평균 Test MAE: 2.9283


In [88]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.098
📈 평균 Test MAE: 3.3571


In [89]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','내수출하지수_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.0929
📈 평균 Test MAE: 2.8803


#AR

In [90]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도qoq.csv')

# GDP 단일 시계열만 사용 (AR(1))
y = df['gdp'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train 예측 ---
        # fittedvalues는 길이가 n-1이므로 첫 번째 값 제외하고 MAE 계산
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 (1분기 앞 예측) ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 2.4989
📈 평균 Test MAE: 2.9953


#XGB

In [92]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','인력사정실적BSI','내수출하지수_lag1']


y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 1.0, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.1511
📈 평균 Test MAE: 2.7623


In [93]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','내수출하지수_lag1']


y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 0.8}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.2524
📈 평균 Test MAE: 3.2791


#LSTM

In [94]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','인력사정실적BSI','내수출하지수_lag1']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 4️⃣ LSTM 입력 형태 변환 함수
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    """
    lookback: 과거 몇 분기 데이터를 사용할지 (ex. 4 = 1년)
    """
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ LSTM 모델 정의 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측
# -----------------------------
train_mae_list, test_mae_list = [], []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    # 윈도우 범위 맞추기
    train_mask = (np.array(idxs) >= train_start) & (np.array(idxs) <= train_end)
    test_mask = (np.array(idxs) == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test, y_test = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X_train, y_train, epochs=200, batch_size=8, verbose=0, callbacks=[early_stop])

        # 예측
        train_pred = model.predict(X_train, verbose=0)
        test_pred = model.predict(X_test, verbose=0)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1))
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 8️⃣ 결과 요약
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 1.0563
📈 평균 Test MAE: 3.8394


#MLP

In [95]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('전기가스수도lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = 'gdp'
exog_vars = ['내수출하지수','자금사정실적BSI_lag1','인건비_lag1','인력사정실적BSI','내수출하지수_lag1'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# 스케일링 (NN에서는 매우 중요)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16), (64,32), (128,64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# 첫 번째 윈도우로 하이퍼파라미터 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=1000)
rand_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y_scaled.loc[train_start:train_end]
    y_test = y_scaled.loc[test_end:test_end]
    X_train = X_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = MLPRegressor(**best_params, random_state=42, max_iter=1000)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1)).flatten()
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'solver': 'adam', 'learning_rate_init': 0.01, 'hidden_layer_sizes': (32, 16), 'alpha': 0.0001, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.0581
📈 평균 Test MAE: 4.1971
